In [32]:
import pandas as pd
path =os.path.join('/data/yongchan/zero-shot-domain-specific-whisper/preprocessed_data/ACTER/en/htfl/annotated/annotations/sequential_annotations/iob_annotations/with_named_entities/htfl_en_001_seq_terms_nes.tsv')
lines = pd.read_csv(path,  sep='\t', skip_blank_lines=False, names=['word', 'boi'])
domain_terms = []
domain_term = []

for i in range(len(lines)):
    candidate_word = lines.iloc[i]['word']
    boi = lines.iloc[i]['boi']
    if boi == 'B':
        domain_term.append(candidate_word)
    elif boi == 'I':
        domain_term.append(candidate_word)
    else:
        if len(domain_term) > 0:
            domain_terms.append(domain_term)
            domain_term = []
            
if len(domain_term) > 0:
    domain_terms.append(domain_term)
    domain_term = []
            
        
print(domain_terms)

[['patient'], ['conditions'], ['patient'], ['Medicare', 'Patient', 'Safety', 'Monitoring', 'System'], ['MPSMS'], ['adverse', 'event'], ['patients', 'hospitalized'], ['acute', 'myocardial', 'infarction'], ['congestive', 'heart', 'failure'], ['pneumonia'], ['conditions'], ['surgery'], ['patients'], ['hospitals'], ['significant'], ['adverse', 'event'], ['acute', 'myocardial', 'infarction'], ['congestive', 'heart', 'failure'], ['in-hospital', 'adverse', 'events'], ['patients'], ['pneumonia'], ['surgical', 'conditions'], ['pressure', 'ulcers'], ['surgical', 'patients'], ['patient']]


# Create dataset with longer text

In [2]:
import os
import re
import numpy as np
import pandas as pd
from IPython.core.debugger import set_trace

train_dataset = pd.DataFrame(columns=['text', 'label', 'unique_label'])
master_path = '/data/yongchan/ATE/ACTER/en'

def normalize_texts(texts):
    text = ' '.join(texts)
    # Remove any empty space that are placed before special characters
    text = re.sub(r'\s+(?=[\W_])', '', text)
    
    # Remove any empty space that are placed after the character /
    text = re.sub(r'/(?=\s)', '/', text)
    return text

for dataset_name in os.listdir(master_path):
    rows = {}
    all_domain_terms = []
    all_texts = []
    all_term_types = []
    dataset_path = os.path.join(master_path, dataset_name, 'annotated')
    
    if dataset_name == 'cor':
        continue
    
    for category in os.listdir(dataset_path):
        category_path = os.path.join(dataset_path, category)
        if category == 'annotations':
            for seq_annotation in os.listdir(category_path):
                if seq_annotation == 'sequential_annotations':
                    iob_annotation_path = os.path.join(category_path, seq_annotation, 'iob_annotations', 'with_named_entities')
                    for filename in sorted(os.listdir(iob_annotation_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                        file_path = os.path.join(iob_annotation_path, filename)
                        lines = pd.read_csv(file_path,  sep='\t', names=['word', 'boi'])
                        domain_terms = []
                        domain_term = []
                        for i in range(len(lines)):
                            candidate_word = lines.iloc[i]['word']                                  
                            boi = lines.iloc[i]['boi']
                            
                            if candidate_word is np.nan and boi is not np.nan:
                                candidate_word = 'None'
                                
                            if boi == 'B':
                                if len(domain_term) == 0:
                                    domain_term.append(candidate_word)
                                else:
                                    domain_term = normalize_texts(domain_term)
                                    domain_terms.append(domain_term)
                                    domain_term = [candidate_word]
                            elif boi == 'I':
                                domain_term.append(candidate_word)
                            else:
                                if len(domain_term) > 0:
                                    domain_term = normalize_texts(domain_term)
                                    domain_terms.append(domain_term)
                                    domain_term = []
                                    
                        if len(domain_term) > 0:
                            domain_term = normalize_texts(domain_term)
                            domain_terms.append(domain_term)
                            domain_term = []
                        all_domain_terms.append(domain_terms)
                        
                elif seq_annotation == 'unique_annotation_lists':
                    unique_annotation_list_path = os.path.join(category_path, seq_annotation, f"{dataset_name}_en_terms_nes.tsv")
                    lines = pd.read_csv(unique_annotation_list_path,  sep='\t', names=['word', 'term_type'])
                    for i, domain_terms in enumerate(all_domain_terms):
                        term_types = []
                        for domain_term in domain_terms:
                            domain_term = domain_term.lower()
                            term_type = ''
                            count = 0
                            while len(term_type) == 0:                                    
                                if count==0:
                                    term_type = lines[lines['word']==domain_term]['term_type']
                                
                                elif count==1:
                                    character_removed_domain_term_wo_ws = re.sub(r'[^a-zA-Z0-9\s]', '', domain_term)
                                    term_type = lines[lines['word']==character_removed_domain_term_wo_ws]['term_type']
                                    
                                elif count==2:
                                    character_removed_domain_term_w_ws = re.sub(r'[^a-zA-Z0-9\s]', ' ', domain_term).split(' ')
                                    for i in sorted(range(len(character_removed_domain_term_w_ws)), reverse=True):
                                        for j in range(len(character_removed_domain_term_w_ws)):
                                            sliced_dash_removed_domain_term_w_ws = ' '.join(character_removed_domain_term_w_ws[j:j+i])
                                            term_type = lines[lines['word'].str.contains(sliced_dash_removed_domain_term_w_ws)]['term_type']
                                            if len(term_type) > 0:
                                                break
                                        if len(term_type) > 0:
                                            break
                                
                                elif count > 2:
                                    term_type = 'Not Annotated'                                            
                                    print(f'Path: {unique_annotation_list_path}, domain_term: {domain_term}, index: {i}')
                                    break
                                    
                                count += 1                                    

                            if not isinstance(term_type, str):
                                term_type = term_type.astype(str).values[0]
                            term_types.append(term_type)
                        all_term_types.append(term_types)
                                
        elif category == 'texts':
            for filename in sorted(os.listdir(category_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                with open(os.path.join(category_path, filename), 'r') as f:
                    text = f.read()
                    all_texts.append(text)
    
    rows = {'text': all_texts, 'label': all_domain_terms, 'unique_label': all_term_types}
    if dataset_name == 'htfl':
        rows['domain'] = ['Heart Failure']*len(all_texts)
        test_dataset = pd.DataFrame(rows)
    elif dataset_name == 'equi':
        rows['domain'] = ['Equitation']*len(all_texts)
        validation_dataset = pd.DataFrame(rows)
    else:
        dataset_name_to_domain = {'corp': 'Corruption', 'wind': 'Wind Energy'}
        rows['domain'] = [dataset_name_to_domain[dataset_name]]*len(all_texts)
        train_dataset = pd.concat([train_dataset, pd.DataFrame(rows)])
        

In [3]:
print(f'Train dataset length: {len(train_dataset)}')
print(f'Validation dataset length: {len(validation_dataset)}')
print(f'Test dataset length: {len(test_dataset)}')

Train dataset length: 17
Validation dataset length: 34
Test dataset length: 190


In [4]:
pd.set_option('display.max_colwidth', 500)

test_dataset.head(10)

,text,label,unique_label,domain
0,"National trends in patient safety for four common conditions, 2005-2011.\nThe effects of more than a decade of national efforts dedicated to improve patient safety remain largely unclear. This study used the Medicare Patient Safety Monitoring System (MPSMS) database to assess national trends in adverse event rates between 2005 through 2011 for patients hospitalized with acute myocardial infarction, congestive heart failure, pneumonia, or conditions requiring surgery. The analysis included a ...","[patient, conditions, patient, Medicare Patient Safety Monitoring System, MPSMS, adverse event, patients, hospitalized, acute myocardial infarction, congestive heart failure, pneumonia, conditions, surgery, patients, hospitals, significant, adverse event, acute myocardial infarction, congestive heart failure, in-hospital, adverse events, patients, pneumonia, surgical, conditions, pressure ulcers, surgical, patients, patient]","[Common_Term, Common_Term, Common_Term, Named_Entity, Named_Entity, Specific_Term, Common_Term, Common_Term, Specific_Term, Specific_Term, Common_Term, Common_Term, Common_Term, Common_Term, Common_Term, OOD_Term, Specific_Term, Specific_Term, Specific_Term, Common_Term, Specific_Term, Common_Term, Common_Term, Common_Term, Common_Term, Specific_Term, Common_Term, Common_Term, Common_Term]",Heart Failure
1,Cross-talk between the heart and adipose tissue in cachectic heart failure patients with respect to alterations in body composition: a prospective study.\nOBJECTIVES:\nCardiac cachexia (CC) is associated with changes in body composition. Lipolysis and increased energy expenditure caused by A- and B natriuretic peptides (NPs) have been suggested to play a role in CC. We tested the hypothesis that neurohormones and adipokines are associated with body composition in CC and that a progressive lo...,"[Cross-talk, heart, adipose tissue, cachectic heart failure, patients, prospective study, Cardiac cachexia, CC, Lipolysis, A-, B natriuretic peptides, NPs, CC, neurohormones, adipokines, CC, fat free mass, FFM, fat mass, FM, FFM, FM, dual energy X-ray absorptiometry, DXA, non-diabetic, patients, chronic heart failure, CHF, CC, non-cachectic CHF, myocardial infarction-both, Biomarkers, neurohormonal stimulation, inflammation, endothelial dysfunction, N-terminal proBNP, NT-proBNP, midregional ...","[Specific_Term, Common_Term, Specific_Term, Specific_Term, Common_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Common_Term, Common_Term, Common_Term, Common_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Common_Term,...",Heart Failure
2,"Diagnosis of heart failure with preserved ejection fraction: role of clinical Doppler echocardiography.\nIn developed countries, at least 38–54% of patients with heart failure show preserved left ventricular (LV) ejection fraction. The prevalence of heart failure with preserved ejection fraction (HFPEF) is steadily increasing and its prognosis is poor. LV diastolic dysfunction, either alone or in combination with other factors (figure 1), is the major underlying mechanism of HFPEF. In the ge...","[Diagnosis, heart failure with preserved ejection fraction, clinical, Doppler echocardiography, patients, heart failure, preserved left ventricular, LV, ejection fraction, prevalence, heart failure with preserved ejection fraction, HFPEF, prognosis, LV diastolic dysfunction, HFPEF, clinical, diastolic dysfunction, mortality, diagnosis, clinical, prognostic, diastolic dysfunction, HFPEF, European Society of Cardiology, HFPEF, symptoms, heart failure, preserved LV ejection fraction, non-dilate...","[Common_Term, Specific_Term, Common_Term, Specific_Term, Common_Term, Common_Term, Specific_Term,

In [5]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_dataset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_dataset, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_dataset, preserve_index=False)

In [6]:
train_dataset['label']

[['HORIZONTAL-AXIS WIND TURBINE BLADES',
  'GRADUATE SCHOOL OF NATURAL AND APPLIED SCIENCES',
  'MIDDLE EAST TECHNICAL UNIVERSITY',
  'SERHAT DURAN',
  'MECHANICAL',
  'ENGINEERING',
  'Graduate School of Natural and Applied Sciences',
  'Canan ÖZGEN',
  'Kemal IDER',
  'Tahsin ÇETINKAYA',
  'O. Cahit ERALP',
  'Kahraman ALBAYRAK',
  'Cemil YAMALI',
  'Kahraman ALBAYRAK',
  'Tahsin ÇETINKAYA',
  'Sinan AKMANDOR',
  'Serhat DURAN',
  'HORIZONTAL-AXIS WIND TURBINE BLADES',
  'Duran',
  'Serhat',
  'Mechanical',
  'Engineering',
  'Kahraman ALBAYRAK',
  'Tahsin ÇETINKAYA',
  'horizontal-axis wind turbine',
  'HAWT',
  'blades',
  'aerodynamic forces',
  'blades',
  'HAWT blade design',
  'aerodynamic',
  'aerodynamic',
  'HAWTs',
  'Blade-element momentum theory',
  'BEM',
  'strip theory',
  'aerodynamic',
  'HAWT blades',
  'HAWT blade design',
  'blade design',
  'rotor',
  'BEM theory',
  'blade shape',
  'blade',
  'loaded',
  'loaded',
  'blade',
  'power prediction',
  'blade',
  '

In [123]:
from datasets import Dataset, DatasetDict

dataset_save_path = '/data/yongchan/ATE/ACTER/huggingface/long'
os.makedirs(dataset_save_path, exist_ok=True)

dataset = DatasetDict({'train': train_dataset, 'validation': validation_dataset, 'test': test_dataset})
dataset.save_to_disk(dataset_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/17 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/190 [00:00<?, ? examples/s]

# Create dataset with shorter text

In [11]:
import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.core.debugger import set_trace

train_dataset = pd.DataFrame(columns=['text', 'label', 'unique_label'])
master_path = '/data/yongchan/ATE/ACTER/en'

def normalize_texts(texts):
    text = ' '.join(texts)
    # Remove any empty space that are placed before special characters
    text = re.sub(r'\s+(?=[\W_])', '', text)
    
    # Remove any empty space that are placed after the character /
    text = re.sub(r'/(?=\s)', '/', text)
    return text

def remove_special_characters(text):
    text = text.lower().replace(' ', '')
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

for dataset_name in os.listdir(master_path):
    rows = {}
    all_domain_terms = []
    all_candidate_words = []
    all_texts = {}
    all_term_types = []
    dataset_path = os.path.join(master_path, dataset_name, 'annotated')
    
    if dataset_name == 'cor':
        continue
    
    for category in sorted(os.listdir(dataset_path)):
        category_path = os.path.join(dataset_path, category)
        
        if category == 'annotations':
            for seq_annotation in sorted(os.listdir(category_path)):                                
                if seq_annotation == 'sequential_annotations':
                    iob_annotation_path = os.path.join(category_path, seq_annotation, 'iob_annotations', 'with_named_entities')
                    for filename in sorted(os.listdir(iob_annotation_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                        file_path = os.path.join(iob_annotation_path, filename)
                        lines = pd.read_csv(file_path,  sep='\t', na_values=[], skip_blank_lines=False, names=['word', 'boi'])
                    
                        split_indexes = lines[pd.isna(lines[['word', 'boi']]).all(axis=1)].index
                        for prev_index, split_index in zip([-1]+list(split_indexes[:-1]), split_indexes):
                            split_lines = lines.iloc[prev_index+1:split_index]
                            domain_terms = []
                            domain_term = []
                            candidate_words = []
                            for i in range(len(split_lines)):
                                candidate_word = split_lines.iloc[i]['word']                                    
                                boi = split_lines.iloc[i]['boi']
                                if candidate_word is np.nan and boi is not np.nan:
                                    candidate_word = 'None'
                                
                                if boi == 'B':
                                    if len(domain_term) == 0:
                                        domain_term.append(candidate_word)
                                    else:
                                        domain_term = normalize_texts(domain_term)
                                        domain_terms.append(domain_term)
                                        domain_term = [candidate_word]
                                elif boi == 'I':
                                    domain_term.append(candidate_word)
                                else:
                                    if len(domain_term) > 0:
                                        domain_term = normalize_texts(domain_term)
                                        domain_terms.append(domain_term)
                                        domain_term = []
                                candidate_words.append(candidate_word.lower())
                            
                            if len(domain_term) > 0:
                                domain_term = normalize_texts(domain_term)
                                domain_terms.append(domain_term)
                                                                    
                            all_domain_terms.append(domain_terms)
                            # all_candidate_words.append(candidate_words)
                            all_candidate_words.append(remove_special_characters(''.join(candidate_words)))
                            
                elif seq_annotation == 'unique_annotation_lists':
                    unique_annotation_list_path = os.path.join(category_path, seq_annotation, f"{dataset_name}_en_terms_nes.tsv")
                    lines = pd.read_csv(unique_annotation_list_path,  sep='\t', names=['word', 'term_type'])
                    for i, domain_terms in enumerate(all_domain_terms):
                        term_types = []
                        for domain_term in domain_terms:
                            domain_term = domain_term.lower()
                            term_type = ''
                            count = 0
                            while len(term_type) == 0:                                    
                                if count==0:
                                    term_type = lines[lines['word']==domain_term]['term_type']
                                
                                elif count==1:
                                    character_removed_domain_term_wo_ws = re.sub(r'[^a-zA-Z0-9\s]', '', domain_term)
                                    term_type = lines[lines['word']==character_removed_domain_term_wo_ws]['term_type']
                                    
                                elif count==2:
                                    character_removed_domain_term_w_ws = re.sub(r'[^a-zA-Z0-9\s]', ' ', domain_term).split(' ')
                                    for i in sorted(range(len(character_removed_domain_term_w_ws)), reverse=True):
                                        for j in range(len(character_removed_domain_term_w_ws)):
                                            sliced_dash_removed_domain_term_w_ws = ' '.join(character_removed_domain_term_w_ws[j:j+i])
                                            term_type = lines[lines['word'].str.contains(sliced_dash_removed_domain_term_w_ws)]['term_type']
                                            if len(term_type) > 0:
                                                break
                                        if len(term_type) > 0:
                                            break
                                
                                elif count > 2:
                                    term_type = 'Not Annotated'                                            
                                    print(f'Path: {unique_annotation_list_path}, domain_term: {domain_term}, index: {i}')
                                    break
                                    
                                count += 1                                    

                            if not isinstance(term_type, str):
                                term_type = term_type.astype(str).values[0]
                            term_types.append(term_type)
                        all_term_types.append(term_types)
        
        # TODO: 여기 고치기...
        elif category == 'texts':
            i =0
            for filename in sorted(os.listdir(category_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                print(f'###### Filename: {filename} is processing... #######')
                with open(os.path.join(category_path, filename), 'r') as f:
                    text = f.read()
                    texts = text.split('\n')                            
                    texts_copy = texts.copy()
                    for text_idx in range(len(texts)):
                        while '\n' in texts[text_idx]:
                            texts[text_idx] = texts[text_idx].replace('\n', '')
                    
                    while '' in texts:
                        texts.remove('')
                        
                    modified_texts = list(map(remove_special_characters, texts))
                    modified_texts_to_texts={modified_text: text for modified_text, text in zip(modified_texts, texts)}
                    temp_all_texts = []
                    temp_all_domain_terms = []
                    temp_all_term_types = []
                    j=0
                    while i < len(all_candidate_words):
                        all_candidate_word = all_candidate_words[i]
                        # if j==286 and filename == 'wind_en_02.txt':
                        #     print('idk')
                        for idx, modified_text in enumerate(modified_texts[j:]):
                            if all_candidate_word in modified_text:
                                temp_all_texts.append(modified_text)
                                domain_term = all_domain_terms[i]
                                temp_all_domain_terms.append(domain_term)
                                term_types = all_term_types[i]
                                temp_all_term_types.append(term_types)
                                if all_candidate_word == modified_text:
                                    j+=1
                                elif i < len(all_candidate_words)-1 and all_candidate_words[i+1] not in modified_text:
                                    j+=1
                                break
                            elif len(modified_text[j:]) <= 1:
                                j+=1
                        
                        i+=1
                        if len(modified_texts[j:]) == 0 or idx == len(modified_texts[j:]) and modified_texts[j:][-1] == all_candidate_word:
                            for modified_text, domain_terms, term_types in zip(temp_all_texts, temp_all_domain_terms, temp_all_term_types):
                                text = modified_texts_to_texts[modified_text]
                                if text not in all_texts:
                                    all_texts[text] = [[],[]]
                                all_texts[text][0].extend(domain_terms)
                                all_texts[text][1].extend(term_types)
                            for text in texts:
                                if text not in list(all_texts.keys()):
                                    print(f'Text: {text} is not in all_texts')
                            print(f'###### Filename: {filename} is finished! #######\n')
                            break
                        
            # if len_texts != len(all_texts):
            #     raise ValueError(f'Length of texts: {len_texts} is not equal to length of all_texts: {len(all_texts)}')
    
    all_text_values = all_texts.values()
    all_domain_terms, all_term_types = [], []
    for all_text_value in all_text_values:
        all_domain_terms.append(all_text_value[0])
        all_term_types.append(all_text_value[1])
        
    rows = {'text': list(all_texts.keys()), 'label': all_domain_terms, 'unique_label': all_term_types}
    rows['dataset_name'] = [dataset_name]*len(all_texts)
    if dataset_name == 'htfl':
        rows['domain'] = ['Heart Failure']*len(all_texts)
        test_dataset = pd.DataFrame(rows)
    elif dataset_name == 'equi':
        rows['domain'] = ['Equitation']*len(all_texts)
        validation_dataset = pd.DataFrame(rows)
    else:
        dataset_name_to_domain = {'corp': 'Corruption', 'wind': 'Wind Energy'}
        rows['domain'] = [dataset_name_to_domain[dataset_name]]*len(all_texts)
        dataset_name_to_domain = {'corp': 'Corruption', 'wind': 'Wind Energy'}
        train_dataset = pd.concat([train_dataset, pd.DataFrame(rows)])
        

###### Filename: wind_en_01.txt is processing... #######
Text: : is not in all_texts
Text: x is not in all_texts
Text: λ = 10 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . is not in all_texts
Text: CP is not in all_texts
Text: P is not in all_texts
Text: w′ is not in all_texts
Text: T is not in all_texts
Text: B is not in all_texts
Text: a is not in all_texts
Text: λ is not in all_texts
Text: λd λr is not in all_texts
Text: ci is not in all_texts
Text: ρ is not in all_texts
Text: Ω is not in all_texts
Text: α is not in all_texts
Text: θ is not in all_texts
Text: θi is not in all_texts
Text: ϕ is not in all_texts
Text: σ ν is not in all_texts
Text: γ Γ is not in all_texts
Text: δΓ is not in all_texts
Text: Re is not in all_texts
Text: 6 is not in all_texts
Text: 12 is not in all_texts
Text: (a) is not in all_texts
Text: (a) is not in all_texts
Text: λ= is not in all_texts
Text: CT = is not in all_texts
Text: a= is not in all_texts
Text: P=

In [13]:
print(f'Train dataset length: {len(train_dataset)}')
print(f'Validation dataset length: {len(validation_dataset)}')
print(f'Test dataset length: {len(test_dataset)}')

Train dataset length: 2326
Validation dataset length: 586
Test dataset length: 754


In [14]:
pd.set_option('display.max_colwidth', 500)
test_dataset.tail(10)

,text,label,unique_label,dataset_name,domain
744,The HeartMate 3 left ventricular assist system (LVAS) is intended to provide long-term support to patients with advanced heart failure. The centrifugal flow pump is designed for enhanced hemocompatibility by incorporating a magnetically levitated rotor with wide blood-flow paths and an artificial pulse.,"[HeartMate, left ventricular assist system, LVAS, patients, heart failure, centrifugal flow pump, hemocompatibility, blood-flow, pulse]","[Named_Entity, Specific_Term, Specific_Term, Common_Term, Common_Term, Specific_Term, Specific_Term, Common_Term, Common_Term]",htfl,Heart Failure
745,"The aim of this single-arm, prospective, multicenter study was to evaluate the performance and safety of this LVAS.","[prospective, LVAS]","[Specific_Term, Specific_Term]",htfl,Heart Failure
746,"The primary endpoint was 6-month survival compared with INTERMACS (Interagency Registry for Mechanically Assisted Circulatory Support)-derived performance goal. Patients were adults with ejection fraction ≤ 25%, cardiac index ≤ 2.2 l/min/m(2) without inotropes or were inotrope-dependent on optimal medical management, or listed for transplant.","[primary endpoint, INTERMACS, Interagency Registry for Mechanically Assisted Circulatory Support)-derived, Patients, ejection fraction, cardiac index, inotropes, medical, transplant]","[Specific_Term, Named_Entity, Specific_Term, Common_Term, Specific_Term, Specific_Term, Specific_Term, Common_Term, Common_Term]",htfl,Heart Failure
747,"Fifty patients were enrolled at 10 centers. The indications for LVAS support were bridge to transplantation (54%) or destination therapy (46%). At 6 months, 88% of patients continued on support, 4% received transplants, and 8% died. Thirty-day mortality was 2% and 6-month survival 92%, which exceeded the 88% performance goal. Support with the fully magnetically levitated LVAS significantly reduced mortality risk by 66% compared with the Seattle Heart Failure Model-predicted survival of 78% (...","[patients, LVAS, bridge to transplantation, destination therapy, patients, transplants, mortality, fully magnetically levitated LVAS, significantly, mortality, Seattle Heart Failure Model-predicted, p, adverse events, reoperation, bleeding, driveline, infection, gastrointestinal bleeding, stroke, Rankin Score, pump thrombosis, hemolysis, New York Heart Association classification, 6-min walk test, quality-of-life]","[Common_Term, Specific_Term, Specific_Term, Specific_Term, Common_Term, Common_Term, Common_Term, Specific_Term, OOD_Term, Common_Term, Named_Entity, OOD_Term, Specific_Term, Common_Term, Common_Term, Specific_Term, Common_Term, Specific_Term, Common_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Specific_Term, Common_Term]",htfl,Heart Failure
748,"The results show that the fully magnetically levitated centrifugal-flow chronic LVAS is safe, with high 30-day and 6-month survival rates, a favorable adverse event profile, and improved quality of life and functional status.","[fully magnetically levitated centrifugal-flow chronic LVAS, adverse event, quality of life, functional status]","[Specific_Term, Specific_Term, Common_Term, Specific_Term]",htfl,Heart Failure
749,Tools for Economic Analysis of Patient Management Interventions in Heart Failure Cost-Effectiveness Model: A Web-based program designed to evaluate the cost-effectiveness of disease management programs in heart failure.,"[Tools for Economic Analysis of Patient Management Interventions in Heart Failure Cost-Effectiveness Model, disease, heart failure]","[Named_Entity, Common_Term, Common_Term]",htfl,Heart Failure
750,"Heart failure disease management programs can influence medical resource use and quality-adjusted survival. Because projecting long-term costs and survival is challenging, a consistent and valid approach to extrapolating short-term outcomes would be valuable.","[Heart failure, disease, medical, outcomes]","[Common_Term, Common_Term, C

In [15]:
train_dataset.tail(10)

,text,label,unique_label,dataset_name,domain
1012,The additional division in sub-objectives is done to facilitate the calculation of the breakdown of estimated impact on expenditure in section 3.2.,[expenditure],[Common_Term],corp,Corruption
1013,DA= Differentiated appropriations / DNA= Non-Differentiated Appropriations,"[appropriations, Appropriations]","[Specific_Term, Specific_Term]",corp,Corruption
1014,EFTA: European Free Trade Association.,"[EFTA, European Free Trade Association]","[Named_Entity, Named_Entity]",corp,Corruption
1015,"Candidate countries and, where applicable, potential candidate countries from the Western Balkans.",[Balkans],[Named_Entity],corp,Corruption
1016,"Technical and/or administrative assistance and expenditure in support of the implementation of EU programmes and/or actions (former ""BA"" lines), indirect research, direct research.","[administrative, expenditure, EU, administrative, expenditure, EU]","[Common_Term, Common_Term, Named_Entity, Common_Term, Common_Term, Named_Entity]",corp,Corruption
1017,"Outputs are products and services to be supplied (e.g.: number of student exchanges financed, number of km of roads built, etc.).",[financed],[Common_Term],corp,Corruption
1018,"Through targeted calls for proposals within the EU programme for prevention and fight against crime (call targeting financial and economic crime), civil society organisations will be encouraged to apply for subject specific assessments of Member States' anti-corruption efforts. The exact number and specific target of each contract cannot be predicted at this point.","[EU, crime, financial, economic crime, civil society, anti-corruption, contract]","[Named_Entity, Common_Term, Common_Term, Common_Term, Common_Term, Common_Term, Common_Term]",corp,Corruption
1019,Public procurement procedures. The contractor will be tasked to set up the network of 27 local research correspondents and cover the coordination/logistic aspects.,"[Public procurement procedures, contractor]","[Common_Term, Common_Term]",corp,Corruption
1020,"The decision to set up such programme and the details of its functioning may only be taken/clarified at a later stage, once the preparations for the first Report are more advanced. One possibility for its setting up would be via procurement procedures (i.e. a contractor tasked to organise a number of roughly 5 experience-sharing workshops/events per year). The costs were estimated for approximately 50 participants + 5 speakers, with 1,000 Euros cost/person and an overall 100,000 Euros for or...","[procurement procedures, contractor, costs, cost]","[Common_Term, Common_Term, Common_Term, Common_Term]",corp,Corruption
1021,"CA= Contract Agent; INT= agency staff ("" Intérimaire"") ; JED= "" Jeune Expert en Délégation"" (Young Experts in Delegations); LA= Local Agent; SNE= Seconded National Expert;",[Contract],[Common_Term],corp,Corruption


In [128]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_dataset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_dataset, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_dataset, preserve_index=False)

In [129]:
from datasets import Dataset, DatasetDict

dataset_save_path = '/data/yongchan/ATE/ACTER/huggingface/short'
os.makedirs(dataset_save_path, exist_ok=True)

dataset = DatasetDict({'train': train_dataset, 'validation': validation_dataset, 'test': test_dataset})
dataset.save_to_disk(dataset_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/2326 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/586 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/754 [00:00<?, ? examples/s]